In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date


In [3]:
spark = (
    SparkSession.builder
    .appName("sales-orders-etl")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse")
    .enableHiveSupport()
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 10:40:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
input_path = "hdfs://namenode:9000/data/raw/sales/sales_orders.csv"

spark.read.option("header", True).csv(input_path).show(5)

26/04/25 10:40:46 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------+----------+----------+-------------+-------------------+-----------+--------+----------+------------+
|order_id|order_date|    region|     customer|            product|   category|quantity|unit_price|sales_amount|
+--------+----------+----------+-------------+-------------------+-----------+--------+----------+------------+
|    1001|2026-01-03|Phnom Penh|  Acme Retail|         Laptop Pro|Electronics|       2|    950.00|     1900.00|
|    1002|2026-01-04| Siem Reap|  Golden Mart|     Mouse Wireless|Accessories|      10|     18.50|      185.00|
|    1003|2026-01-05|Battambang|  Mekong Shop|       Office Chair|  Furniture|       4|    120.00|      480.00|
|    1004|2026-01-05|Phnom Penh|Trust Trading|         Monitor 27|Electronics|       3|    210.00|      630.00|
|    1005|2026-01-06|    Kampot| Lotus Market|Keyboard Mechanical|Accessories|       6|     45.00|      270.00|
+--------+----------+----------+-------------+-------------------+-----------+--------+----------+------

In [6]:
input_path = "hdfs://namenode:9000/data/raw/sales/sales_orders.csv"
output_path = "hdfs://namenode:9000/data/curated/sales_orders_parquet"

In [7]:
df = (
    spark.read
    .option("header", True)
    .csv(input_path)
    .withColumn("order_id", col("order_id").cast("int"))
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("sales_amount", col("sales_amount").cast("double"))
)

(
    df.write
    .mode("overwrite")
    .parquet(output_path)
)

In [8]:
spark.sql("CREATE DATABASE IF NOT EXISTS test")
spark.sql("DROP TABLE IF EXISTS lab.sales_orders_curated")
spark.sql(f"""
CREATE EXTERNAL TABLE lab.sales_orders_curated (
  order_id INT,
  order_date DATE,
  region STRING,
  customer STRING,
  product STRING,
  category STRING,
  quantity INT,
  unit_price DOUBLE,
  sales_amount DOUBLE
)
STORED AS PARQUET
LOCATION '{output_path}'
""")

spark.sql("SELECT region, SUM(sales_amount) AS total_sales FROM lab.sales_orders_curated GROUP BY region").show()

spark.stop()

26/04/25 10:41:33 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+----------+-----------+
|    region|total_sales|
+----------+-----------+
| Siem Reap|      449.0|
|Battambang|     1120.0|
|    Kampot|      725.0|
|Phnom Penh|     3425.0|
+----------+-----------+

